# Notebook 09 — Optimizer Backends and Multi-Start Outputs

This notebook explains the multi-start optimisation strategy,
available solver backends, and the structure of the result object
returned after fitting.

> **Note**: We do **not** actually run the optimiser here (too slow for a
> notebook). Instead we build the problem object conceptually and use
> **synthetic data** to illustrate all output formats.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1  Multi-start strategy

A single gradient-based optimiser converges to a local minimum.
Multi-start mitigates this by:

1. Sampling `n_starts` initial `theta_0` uniformly from `[xl, xu]`
2. Running a local optimiser from each starting point (optionally in parallel)
3. Collecting all converged solutions and selecting the best by `min(J)`

```python
# Typical call via CLI (recommended)
# phoscrosstalk --config config.toml

# Or via Python API:
import argparse
from phoscrosstalk.multistarts import run_multi_start_optimization

args = argparse.Namespace(
    n_starts=20, max_steps=500, backend='optimistix',
    ls_solver='lm', jac_mode='fwd',
    parallel_starts=1, threads_per_start=1,
)
merged_res, best_idx, total_losses = run_multi_start_optimization(
    problem,    # NetworkProblem
    args,       # argparse.Namespace
    P_scaled,   # (N, T) scaled phosphosite data
)
theta_best = merged_res.X[best_idx]
```


## 2  Available solver backends

| Backend | Algorithm | Notes |
|---|---|---|
| `optimistix` (default) | Levenberg-Marquardt or BFGS | JAX-native, recommended |
| `jaxopt_lbfgsb` | L-BFGS-B | Good for smooth objectives |
| `jaxopt_pgd` | Projected Gradient Descent | Box constraints explicit |
| `jaxopt_osqp` | OSQP (quadratic program) | Box + linear constraints |
| `jaxopt_box_osqp` | Box-constrained OSQP | Quadratic subproblem |
| `jaxopt_eq_qp` | Equality-constrained QP | Strict equality constraints |
| `scipy_jax` | L-BFGS-B via `scipy.optimize` | Uses JAX gradients; no JIT |

All backend names are validated against `_VALID_BACKENDS` in `config.py` —
any unknown string raises at startup.

**Jacobian mode** (`jac_mode`):
- `'fwd'` \u2014 `jax.jacfwd`: efficient when `n_params << n_residuals`
- `'bwd'` \u2014 `jax.jacrev`: efficient when `n_params >> n_residuals`

**ODE adjoint** must match: `'fwd'` \u2192 `ode_adjoint='forward'`, `'bwd'` \u2192 `'recursive'`


## 3  Key optimisation settings

| Parameter | Default | Meaning |
|---|---|---|
| `n_starts` | 1 | Number of random restarts |
| `max_steps` | 500 | Max LM/BFGS iterations per start |
| `ls_solver` | `'lm'` | `'lm'` or `'bfgs'` (optimistix only) |
| `jac_mode` | `'fwd'` | Forward or reverse AD |
| `parallel_starts` | 1 | Concurrent optimisations |
| `threads_per_start` | 1 | XLA threads per optimisation |
| `reg_lambda` | 0.001 | L2 penalty on theta |
| `lambda_net` | 0.001 | Kinase-network Laplacian penalty |

In [ ]:
from phoscrosstalk.config import ModelDims
from phoscrosstalk.data_loader import (
    load_site_data, load_rna_data, load_kinase_site_matrix,
    load_tf_network, build_tf_prot_weights, apply_scaling, row_normalize,
)

TIMEPOINTS = list(range(1, 15))   # 14 time points x1..x14

# ── phosphosite + protein abundance ─────────────────────────────────────────
sites, proteins, site_prot_idx, positions, t_phos, Y, A_data, A_proteins = \
    load_site_data(str(SAMPLE_DIR / "protephospho.csv"), TIMEPOINTS)

K = len(proteins)
N = len(sites)
T = len(t_phos)
print(f"proteins : {proteins}  (K={K})")
print(f"sites    : {sites}  (N={N})")
print(f"t_phos   : {t_phos}  (T={T})")
print(f"Y        : {Y.shape}   (N × T  phosphosite data)")
print(f"A_data   : {A_data.shape}  (K × T  protein abundance)")

# ── mRNA ─────────────────────────────────────────────────────────────────────
gene_ids, t_rna, rna_matrix = load_rna_data(
    str(SAMPLE_DIR / "mrna.csv"), timepoints=TIMEPOINTS
)
print(f"gene_ids : {gene_ids}  (n_genes={len(gene_ids)})")
print(f"rna_matrix: {rna_matrix.shape}  (n_genes × T)")

# ── kinase-site matrix ───────────────────────────────────────────────────────
K_site_kin, kinases = load_kinase_site_matrix(
    str(SAMPLE_DIR / "kinase_sites.tsv"), sites
)
M = len(kinases)
print(f"kinases  : {kinases}  (M={M})")
print(f"K_site_kin: {K_site_kin.shape}  (N × M)")

# ── kinase → protein index ───────────────────────────────────────────────────
kin_to_prot_idx = np.array([proteins.index(k) for k in kinases], dtype=int)
print(f"kin_to_prot_idx: {kin_to_prot_idx}")

# ── TF network ───────────────────────────────────────────────────────────────
tf_net = load_tf_network(str(SAMPLE_DIR / "tf_mrna.csv"), gene_ids=gene_ids)
tf_prot_weights = build_tf_prot_weights(tf_net, gene_ids, proteins)
print(f"tf_prot_weights: {tf_prot_weights.shape}  (K × n_genes)")

# ── scaled data ──────────────────────────────────────────────────────────────
P_scaled, _, _  = apply_scaling(Y)
A_scaled, _, _  = apply_scaling(A_data)
dims = ModelDims(K=K, M=M, N=N)

In [ ]:
from phoscrosstalk.optimization import create_bounds, build_parameter_labels, NetworkProblem
from phoscrosstalk.weighting import build_weight_matrices
from phoscrosstalk.derived_rates import make_k_act_fn, make_s_prod_fn

Cg = np.zeros((N, N)); Cl = np.zeros((N, N))
R_kin = row_normalize(K_site_kin.T); L_alpha = np.zeros((M, M))
receptor_mask_prot = np.zeros(K); receptor_mask_kin = np.zeros(M)

k_act_fn = make_k_act_fn(t_rna=t_rna, rna_data=rna_matrix,
                         tf_prot_weights=tf_prot_weights, K=K)
s_prod_fn = make_s_prod_fn(
    t_protein=t_phos, Y_data=P_scaled,
    R_kin_site=row_normalize(K_site_kin.T),
    kin_to_prot_idx=kin_to_prot_idx, K=K, M=M,
)
W_data, W_prot, _ = build_weight_matrices(t=t_phos, Y=Y, A_data=A_data)
xl, xu, dim = create_bounds(K, M, N)
print(f"theta dim={dim}  bounds: [{xl.min():.2f}, {xu.max():.2f}]")


## 4  `NetworkProblem` — the problem container

In [ ]:
# Build the NetworkProblem (no optimisation is run here)
# prot_idx_for_A maps each row of A_scaled to its protein index
prot_idx_for_A = np.arange(K, dtype=int)

prob = NetworkProblem(
    dims=dims,
    t=t_phos,
    P_data=P_scaled,
    Cg=Cg,
    Cl=Cl,
    site_prot_idx=np.array(site_prot_idx),
    K_site_kin=K_site_kin,
    R=R_kin,
    A_scaled=A_scaled,
    prot_idx_for_A=prot_idx_for_A,
    W_data=W_data,
    W_data_prot=W_prot,
    L_alpha=L_alpha,
    kin_to_prot_idx=kin_to_prot_idx,
    lambda_net=1e-3,
    reg_lambda=1e-3,
    receptor_mask_prot=receptor_mask_prot,
    receptor_mask_kin=receptor_mask_kin,
    mechanism="dist",
    xl=xl,
    xu=xu,
    k_act_fn=k_act_fn,
    s_prod_fn=s_prod_fn,
)
print("NetworkProblem created")
print("  P_data shape:", prob.P_data.shape)
print("  A_scaled shape:", prob.A_scaled.shape)
print("  K_site_kin:  ", prob.K_site_kin.shape)


## 5  Synthetic multi-start results

The real `run_multi_start_optimization()` returns `(merged_res, best_idx, total_losses)`.
We simulate this structure to show how to work with the outputs.


In [ ]:
from phoscrosstalk.simulation import simulate as _simulate

n_starts = 5
rng = np.random.default_rng(42)
theta_starts = np.array([rng.uniform(xl, xu) for _ in range(n_starts)])

all_F, all_J = [], []
for i, theta_i in enumerate(theta_starts):
    P_sim_i, A_sim_i = _simulate(
        t_arr=t_phos, P_data0=P_scaled, A_data0=A_scaled,
        theta=theta_i, Cg=Cg, Cl=Cl,
        site_prot_idx=site_prot_idx, K_site_kin=K_site_kin, R=R_kin,
        L_alpha=L_alpha, kin_to_prot_idx=kin_to_prot_idx,
        receptor_mask_prot=receptor_mask_prot, receptor_mask_kin=receptor_mask_kin,
        mechanism="dist", k_act_fn=k_act_fn, s_prod_fn=s_prod_fn,
    )
    f1 = float(np.nanmean(W_data * (P_sim_i - P_scaled)**2))
    f2 = float(np.nanmean(W_prot * (A_sim_i - A_scaled)**2))
    f3 = float(1e-3 * np.sum(theta_i**2))
    f4 = 0.0
    J_i = f1 + f2 + f3 + f4
    all_F.append([f1, f2, f3, f4])
    all_J.append(J_i)
    print(f"  Start {i}: f1={f1:.4f}  f2={f2:.4f}  f3={f3:.4f}  J={J_i:.4f}")

all_F = np.array(all_F)
all_J = np.array(all_J)
best_idx = int(np.argmin(all_J))
print(f"\nBest start: {best_idx}  (J={all_J[best_idx]:.4f})")


## 6  Result structure

| Attribute | Shape | Content |
|---|---|---|
| `merged_res.X` | `(n_starts, dim)` | All converged theta vectors |
| `merged_res.F` | `(n_starts, 4)` | Loss components `[f1, f2, f3, f4]` |
| `merged_res.J` | `(n_starts,)` | Total loss per start |

`best_idx = np.argmin(merged_res.J)` → `theta_best = merged_res.X[best_idx]`

In [ ]:
from types import SimpleNamespace
merged_res = SimpleNamespace(X=theta_starts, F=all_F, J=all_J)

theta_best = merged_res.X[best_idx]
print("theta_best shape:", theta_best.shape)
print("Best start losses:", merged_res.F[best_idx].round(6))

pareto_df = pd.DataFrame(merged_res.F, columns=["f1_phospho","f2_protein","f3_reg","f4_mrna"])
pareto_df["J_total"] = merged_res.J
pareto_df.index.name = "start"
print("\nPareto table:")
print(pareto_df.round(5).to_string())


## 7  Convergence plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ["tab:orange" if i == best_idx else "steelblue" for i in range(n_starts)]
axes[0].bar(range(n_starts), merged_res.J, color=colors)
axes[0].set_xlabel("Start index"); axes[0].set_ylabel("Total loss J")
axes[0].set_title("Total loss per start  (orange = best)")
axes[0].set_xticks(range(n_starts))

bottoms = np.zeros(n_starts)
comp_colors = ["tab:blue","tab:orange","tab:green","tab:red"]
comp_labels = ["f1 phospho","f2 protein","f3 reg","f4 mRNA"]
for c_idx, (col, lbl) in enumerate(zip(comp_colors, comp_labels)):
    axes[1].bar(range(n_starts), merged_res.F[:, c_idx],
                bottom=bottoms, color=col, label=lbl, alpha=0.85)
    bottoms += merged_res.F[:, c_idx]
axes[1].set_xlabel("Start index"); axes[1].set_ylabel("Loss contribution")
axes[1].set_title("Stacked loss components per start")
axes[1].set_xticks(range(n_starts)); axes[1].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_multistart_convergence.png", dpi=100)
plt.show()
print("Saved 09_multistart_convergence.png")


## 8  Save pareto_front_with_J.tsv

In [ ]:
pareto_out = pareto_df.copy()
pareto_out.insert(0, "start_idx", range(n_starts))
pareto_out["frechet_dist"] = np.nan  # filled after real optimisation
out_path = OUTPUT_DIR / "09_pareto_front_with_J.tsv"
pareto_out.to_csv(out_path, sep="\t", index=False)
print("Saved:", out_path)
print(pareto_out.to_string(index=False))


## 9  Fréchet distance diagnostic

After optimisation, **Fréchet distance** measures curve similarity between
simulated and observed trajectories as an alternative to MSE:

- MSE is point-wise; Fréchet distance captures **path shape** similarity
- A small Fréchet distance means the simulated trajectory follows the same
  shape as the data, not just similar values at matching time points
- Computed via `phoscrosstalk.fretchet` (discrete Fréchet algorithm)
- Stored in `pareto_front_with_J.tsv` for post-hoc Pareto analysis


## 10  CPU parallelism settings

Multi-start optimisation parallelises at two levels:

| Setting | Effect |
|---|---|
| `parallel_starts` | Runs N optimisations concurrently via `concurrent.futures` |
| `threads_per_start` | Sets XLA thread count per optimisation (`XLA_FLAGS`) |

**Rule of thumb**: `parallel_starts × threads_per_start ≤ n_cpu_cores`

Example for a 16-core machine with 20 starts:
```python
parallel_starts=4, threads_per_start=4, n_starts=20
```